In [ ]:
import sys
import os
import yaml
from langchain_core.documents import Document

# 1. 경로 설정 및 모듈 임포트
PROJECT_ROOT = "/home/taehoon/sprint-ai-mid-project_team3"
if os.path.abspath(PROJECT_ROOT) not in sys.path:
    sys.path.append(os.path.abspath(PROJECT_ROOT))

# 함수(create_retriever)를 직접 불러옵니다.
from src.retriever_factory import create_retriever

# default.yaml 설정 파일을 직접 로드합니다.
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config/default.yaml")
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"설정 파일이 존재하지 않습니다: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config = yaml.safe_load(f)

retrieval_config = full_config.get("retrieval") or full_config.get("retriever", {})

# 2. 지정된 경로에서 sample_rfp.txt 파일을 읽어와 청킹 수행
FILE_PATH = os.path.join(PROJECT_ROOT, "samples/raw/sample_rfp.txt")
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"지정한 경로에 파일이 존재하지 않습니다: {FILE_PATH}")

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# [전처리] 텍스트를 문단(연속된 두 줄 줄바꿈) 단위로 분할하여 청크 생성
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

# [해결 포인트 1] 불필요하게 Document 객체를 중복 생성하던 processed_docs = [] 리스트 전체 제거
chunks_for_factory = []

for idx, paragraph in enumerate(paragraphs):
    if idx % 2 == 0:
        org = "과학기술정보통신부"
        project = "지능형_AI_플랫폼_구축"
    else:
        org = "산업통상자원부"
        project = "에너지_빅데이터_분석"
    
    # 팩토리의 chunks 인자로 들어갈 딕셔너리에 메타데이터 정보까지 통합하여 빌드
    chunks_for_factory.append({
        "text": paragraph, 
        "org_name": org, 
        "project_name": project,
        "doc_id": f"DOC_RFP_{idx:03d}"
    })
        
print(f"총 {len(chunks_for_factory)}개의 텍스트 문서 청크가 준비되었습니다.")


# 3. [해결 포인트 2] 리트리버를 생성하면서 초기에 데이터를 주입하거나 내부에서 처리하도록 일임
retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)
print("Chroma DB에 sample_rfp 데이터 적재 프로세스 완료!\n")


# 4. [해결 포인트 3] 규격 변경 반영 (retrieve -> search / page_content -> text)
query_str = "사업의 추진 배경 및 기대 효과를 알려주세요"

print("=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===")
results = retriever.search(query=query_str, top_k=3)
for d in results:
    print(f"[{d.metadata['doc_id']} / {d.metadata['org_name']}](Score: {d.score:.4f}) {d.text[:60]}...")

print("\n=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===")
results_filtered = retriever.search(query=query_str, org_name="과학기술정보통신부", top_k=2)
for d in results_filtered:
    print(f"[{d.metadata['org_name']} / {d.metadata['project_name']}](Score: {d.score:.4f}) {d.text[:60]}...")

print("\n=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===")
results_doc = retriever.search(query=query_str, doc_id="DOC_RFP_001")
if results_doc:
    print(f"[{results_doc[0].metadata['doc_id']}](Score: {results_doc[0].score:.4f}) {results_doc[0].text}")


print("\n" + "="*50)
print("=== 5. (추가) Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===")
print("="*50)

# 1. 과기부 전용 리트리버 생성
msit_retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)

# 2. 산자부 전용 리트리버 생성
motie_retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)

# 3. 검색 시 규격 반영 (retrieve -> search)
results_hybrid = msit_retriever.search(query="보안 기술", project_name="AI보안사업")
print("Multi-Collection 하이브리드 테스트 호출 완료")

총 16개의 텍스트 문서 청크가 준비되었습니다.


/home/taehoon/sprint-ai-mid-project_team3/src/local_chroma_retriever.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/home/taehoon/sprint-ai-mid-project_team3/src/local_chroma_retriever.py:38: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_store = Chroma(


Chroma DB에 sample_rfp 데이터 적재 프로세스 완료!

=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===

=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===

=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===

=== 5. (추가) Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Multi-Collection 하이브리드 테스트 호출 완료
